In [13]:
import osmnx as ox
import pandas as pd
import os
import networkx as nx
import matplotlib.pyplot as plt
import geopy.distance
import numpy as np
import geopandas as gpd

In [14]:
#4 representative clusters
ox.settings.use_cache = True
clusters = {
    "urban_honolulu":{ "cluster_coords":(21.304547, -157.855676), },  
    "suburban_mililani": {"cluster_coords":(21.4508308, -158.0095783),},
    "rural_waianae": { "cluster_coords":(21.4454875, -158.1878239),},
    "remote_ocean_view":{"cluster_coords":(19.1033522, -155.785762),}                          
}

In [15]:
cai_approved=pd.read_excel("../data/raw/cai_approved.xlsx")

In [16]:
for name, data in clusters.items():
    filepath = f"../data/raw/{name}_network.graphml"
    if os.path.exists(filepath):
        G=ox.load_graphml(filepath)
        print(f"{name} already exists")
    else:
        G = ox.graph_from_point(data["cluster_coords"], dist=8000, network_type="drive")
        ox.save_graphml(G,filepath)
        print(f"{name} saved.")  

urban_honolulu already exists
suburban_mililani already exists
rural_waianae already exists
remote_ocean_view already exists


In [17]:
#take closest 3 cais, sort from smallest to largest
cluster_cais={}
for name, data in clusters.items():
    cluster_center = data["cluster_coords"]
    df_temp = cai_approved.copy()
    df_temp["cluster_distance"] = df_temp.apply(lambda row: geopy.distance.geodesic((row["latitude"], row["longitude"]),cluster_center).km,
    axis=1)
    cluster_cais[name]=df_temp.nsmallest(3,"cluster_distance")

In [18]:
#distance between every road node to the road node from the nearest cai
for name, data in clusters.items():
    filepath = f"../data/raw/{name}_network.graphml"
    G = ox.load_graphml(filepath)
    target_df=cluster_cais[name]
    cai_nodes = [ox.nearest_nodes(G, X=row['longitude'],Y=row['latitude']) 
    for _, row in target_df.iterrows()]
    distances = nx.multi_source_dijkstra_path_length(G, list(set(cai_nodes)), weight='length')
    nx.set_node_attributes(G, distances, "dist_to_cai_proxy")
    ox.save_graphml(G, filepath=f"../data/raw/{name}_network_with_distance.graphml")
    print(f"saved {name} network with distance.")   

saved urban_honolulu network with distance.
saved suburban_mililani network with distance.
saved rural_waianae network with distance.
saved remote_ocean_view network with distance.


In [26]:
#summary stats for calculated proxy distances
for name in clusters:
    G_test = ox.load_graphml(f"../data/raw/{name}_network_with_distance.graphml")
    distances = [
        float(data["dist_to_cai_proxy"])
        for node, data in G_test.nodes(data=True)
        if "dist_to_cai_proxy" in data
    ]
    print(
        name,
        "mean:", np.mean(distances),
        "median:", np.median(distances),
        "max:", np.max(distances)
    )

urban_honolulu mean: 6418.885291644303 median: 5987.913590057511 max: 19761.415549791724
suburban_mililani mean: 6056.162250752098 median: 5574.657507544312 max: 18919.08770095325
rural_waianae mean: 7464.649067915268 median: 7030.74474488872 max: 17704.910390543842
remote_ocean_view mean: 7359.273380878038 median: 7221.852092086492 max: 12759.28865123686


The cell below contains pre-calculated terrain statistics, followed by calculations for a single cluster, hence the commented out code per cluster in the cell below.

In [27]:
terrain_scores = {
    "urban_honolulu": {"terrain_p75": 0.0435},
    "suburban_mililani": {"terrain_p75": 0.0328},
    "rural_waianae": {"terrain_p75": 0.0242},
    "remote_ocean_view": { "terrain_p75": 0.0845}
}
#name="urban_honolulu"
#name="suburban_mililani"
#name="rural_waianae"
name="remote_ocean_view"
G=ox.load_graphml(f"../data/raw/{name}_network_with_distance.graphml")

In [28]:
#add elevation  
G= ox.elevation.add_node_elevations_raster( G, "../data/raw/n20w1562023.tif")
#calcuate road slope
G = ox.elevation.add_edge_grades(G)
grades = [
    data["grade_abs"]
    for _, _, data in G.edges(data=True)
    if "grade_abs" in data
]
terrain_scores[name] = {
    "median": np.percentile(grades, 50),
    "p75": np.percentile(grades, 75),
    "p90": np.percentile(grades, 90)
}
ox.save_graphml(G, f"../data/raw/{name}_network_with_terrain.graphml")
print(terrain_scores)

{'urban_honolulu': {'terrain_p75': 0.0435}, 'suburban_mililani': {'terrain_p75': 0.0328}, 'rural_waianae': {'terrain_p75': 0.0242}, 'remote_ocean_view': {'median': np.float64(0.06964641427203747), 'p75': np.float64(0.08451631418559667), 'p90': np.float64(0.1005213354602686)}}


In [29]:
housing = pd.read_csv("../data/raw/acs_data/housing.csv", skiprows=1)
housing.head()

,Geography,Geographic Area Name,Estimate!!Total,Margin of Error!!Total,Unnamed: 4
0,1500000US150010201001,Block Group 1; Census Tract 201; Hawaii County...,627,111,NaN
1,1500000US150010201002,Block Group 2; Census Tract 201; Hawaii County...,354,96,NaN
2,1500000US150010201003,Block Group 3; Census Tract 201; Hawaii County...,700,171,NaN
3,1500000US150010201004,Block Group 4; Census Tract 201; Hawaii County...,514,106,NaN
4,1500000US150010202021,Block Group 1; Census Tract 202.02; Hawaii Cou...,336,94,NaN


In [30]:
#removing duplicate header row and rename columns
housing = housing[housing["Geography"] != "Geography"]
housing = housing.rename(columns={
    "Geography": "GEO_ID",
    "Estimate!!Total": "housing"
})
#takes out prefix from every geo id
housing["GEOID"] = housing["GEO_ID"].str.replace(
    "1500000US",
    "",
    regex=False
)
housing["GEOID"].head()

0    150010201001
1    150010201002
2    150010201003
3    150010201004
4    150010202021
Name: GEOID, dtype: str

In [31]:
census = gpd.read_file(
    "../data/raw/census data/tl_2021_15_bg.shp"
)
#merges geo id and housing count with census dataset
census = census.merge(
    housing[["GEOID", "housing"]],
    on="GEOID",
    how="left"
)
#converts housing to a float/numeric data type
census["housing"] = pd.to_numeric(
    census["housing"],
    errors="coerce"
)
print(census[["GEOID", "housing"]].head())

          GEOID  housing
0  150030103081      553
1  150030065002      238
2  150010215104      766
3  150010215101      874
4  150010215111      689


In [32]:
#housing density value for every census block group in Hawaii
census["area_km2"] = census["ALAND"] / 1e6
census["housing_density"] = census["housing"] / census["area_km2"]
census["housing_density"].describe()

count     1058.000000
mean      2261.077449
std       4797.563880
min          0.000000
25%        202.609108
50%        867.201257
75%       1800.182919
max      57417.772467
Name: housing_density, dtype: float64

In [33]:
census.head()

,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,GEOID,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry,housing,area_km2,housing_density
0,15,003,010308,1,150030103081,Block Group 1,G5030,S,872791,0,+21.4034045,-157.8081196,"POLYGON ((-157.81581 21.40262, -157.81556 21.4...",553,0.872791,633.599567
1,15,003,006500,2,150030065002,Block Group 2,G5030,S,157343,0,+21.3512718,-157.8578121,"POLYGON ((-157.8603 21.35032, -157.86008 21.35...",238,0.157343,1512.618928
2,15,001,021510,4,150010215104,Block Group 4,G5030,S,2940948,4042806,+19.5778245,-155.9675565,"POLYGON ((-155.97913 19.60056, -155.97886 19.6...",766,2.940948,260.460233
3,15,001,021510,1,150010215101,Block Group 1,G5030,S,1949149,0,+19.5955637,-155.9658709,"POLYGON ((-155.97432 19.59925, -155.97335 19.5...",874,1.949149,448.400815
4,15,001,021511,1,150010215111,Block Group 1,G5030,S,3507435,0,+19.5831169,-155.9483577,"POLYGON ((-155.96135 19.6021, -155.96102 19.60...",689,3.507435,196.439848


In [50]:
#transfer density values to network nodes
def add_density_to_nodes(G, census):
    nodes, edges = ox.graph_to_gdfs(G)
    census_proj = census.to_crs(nodes.crs)
    
    nodes = gpd.sjoin(
        nodes,
        census_proj[["geometry", "housing_density"]],
        how="left",
        predicate="within"
    )
    
    nodes["housing_density"] = nodes["housing_density"].fillna(0)
    return nodes, edges

In [51]:
nodes_dict = {}
edges_dict = {}

for name in clusters:
    print(name)
    G = ox.load_graphml(f"../data/raw/{name}_network_with_terrain.graphml")
    nodes, edges = ox.graph_to_gdfs(G)
    nodes = add_density_to_nodes(G,census)[0]
    edges = add_density_to_edges( nodes,edges)

    nodes_dict[name] = nodes
    edges_dict[name] = edges

    display(edges[["length","grade", "housing_density"]].head())
    nodes.to_file(f"../data/processed/{name}_nodes_with_features.gpkg", driver="GPKG")
    edges.to_file(f"../data/processed/{name}_edges_with_features.gpkg", driver="GPKG")

urban_honolulu


,length,grade,housing_density
0,134.045222,0.004770,923.980378
1,938.546530,0.000373,923.980378
2,85.010583,-0.109088,6320.513002
3,176.388600,0.127452,6320.513002
4,320.328055,0.005004,9829.614460


suburban_mililani


,length,grade,housing_density
0,91.473537,0.009825,1036.952751
1,45.048991,0.026894,1036.952751
2,46.276899,-0.027748,1036.952751
3,91.473537,-0.009825,1036.952751
4,41.352138,0.000911,306.356296


rural_waianae


,length,grade,housing_density
0,32.635771,-0.005646,19.417059
1,224.955732,-0.005830,1135.032777
2,249.408658,-0.002501,1048.711163
3,46.590813,-0.023356,438.774697
4,185.799065,-0.006067,308.221628


remote_ocean_view


,length,grade,housing_density
0,216.632106,0.009837,32.884580
1,3769.659100,-0.071875,27.227485
2,503.742909,0.003679,45.933310
3,400.257649,-0.091007,19.835850
4,400.976979,0.074039,19.835850


In [52]:
nodes_dict = {}
edges_dict = {}

for name in clusters:

    #load terrain and distance networs
    print("PROCESSING:", name)
    G_terrain = ox.load_graphml(f"../data/raw/{name}_network_with_terrain.graphml")
    nodes, edges = ox.graph_to_gdfs( G_terrain)
    G_distance = ox.load_graphml(f"../data/raw/{name}_network_with_distance.graphml")
    nodes_distance, _ = ox.graph_to_gdfs( G_distance)
    
    #assign cai_proxyy distance to nodes
    nodes["dist_to_cai_proxy"] = nodes_distance["dist_to_cai_proxy"]
    nodes["dist_to_cai_proxy"] = pd.to_numeric(nodes["dist_to_cai_proxy"],errors="coerce")

    #transfer node distances to edges through endpoints u and v
    edges = edges.reset_index()
    edges["dist_to_cai_proxy_u"] = edges["u"].map(nodes["dist_to_cai_proxy"] )
    edges["dist_to_cai_proxy_v"] = edges["v"].map(nodes["dist_to_cai_proxy"])
    edges["dist_to_cai_proxy"] = (edges["dist_to_cai_proxy_u"] + edges["dist_to_cai_proxy_v"])/ 2
    
    nodes_dict[name] = nodes
    edges_dict[name] = edges

    print(edges[ ["length", "grade","dist_to_cai_proxy" ] ].head())

PROCESSING: urban_honolulu
       length     grade  dist_to_cai_proxy
0  134.045222  0.004770        5330.012399
1  938.546530  0.000373        5732.263052
2   85.010583 -0.109088        3113.931407
3  176.388600  0.127452        3244.630999
4  320.328055  0.005004        3316.600726
PROCESSING: suburban_mililani
      length     grade  dist_to_cai_proxy
0  91.473537  0.009825        2347.435270
1  45.048991  0.026894        2324.222997
2  46.276899 -0.027748        2278.560052
3  91.473537 -0.009825        2347.435270
4  41.352138  0.000911       12260.734596
PROCESSING: rural_waianae
       length     grade  dist_to_cai_proxy
0   32.635771 -0.005646       10632.654573
1  224.955732 -0.005830       11378.749296
2  249.408658 -0.002501        8080.738062
3   46.590813 -0.023356        6592.328054
4  185.799065 -0.006067        6708.522993
PROCESSING: remote_ocean_view
        length     grade  dist_to_cai_proxy
0   216.632106  0.009837        7114.581199
1  3769.659100 -0.071875       

In [14]:
for name in clusters:
    nodes_dict[name].to_file(f"../data/processed/{name}_nodes_final.gpkg",driver="GPKG")
    edges_dict[name].to_file(f"../data/processed/{name}_edges_final.gpkg",driver="GPKG")